# Model Evaluation and Predictions

In [1]:
# Import libraries
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import random

import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
# Load Data

#dataset1 = 'Dataset_RF_Model_SW_COMIDS.csv'
dataset1 = 'Dataset_RF_Model_GW_COMIDS.csv'

#modelpath = "/torch_models_kfold_future_NO3_sw.pth"
modelpath = "/torch_models_kfold_future_NO3_gw.pth"

#testnames = "/test_datasets_sw.pkl"
testnames = "/test_datasets_gw.pkl"



In [3]:
# Load Observation Dataset
future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3/'
model_dir = future_dir + "/Models"


input_data_ = pd.read_csv(future_dir+dataset1)
input_data = input_data_.drop(columns=['COMID','viol_freq']) 
#y = input_data['Viol_Class'].values
#ncols = input_data.shape[1]
#ncols
input_data.shape


(115843, 14)

In [4]:
## Save Preditor Data for SHAP Analysis
from torch.utils.data import DataLoader, TensorDataset, SubsetRandomSampler

X_input_data = input_data.drop(columns=['Viol_Class']).values
X_input_data.shape

#  Convert to tensor data
X_input_data = torch.from_numpy(X_input_data).type(torch.float)
X_input_data.shape,X_input_data.dtype

# Find the size of the smallest class
min_size = input_data['Viol_Class'].value_counts().min()

min_size  = min_size * 10

# Sample exactly 'min_size' elements from each binary group
#balanced_df = input_data.groupby('Viol_Class').sample(n=min_size, random_state=42).reset_index(drop=True)

# If increasing the min_size, you can use the 'replace=True' argument to allow for sampling with replacement
balanced_df = input_data.groupby('Viol_Class').sample(n=min_size, random_state=42, replace=True).reset_index(drop=True)


# Convert to tensors and split into train and test sets
from sklearn.model_selection import train_test_split
#X = input_data.drop(columns=['Viol_Class']).values # when use this the model just predicts the majority class
#y = input_data['Viol_Class'].values
X = balanced_df.drop(columns=['Viol_Class']).values
y = balanced_df['Viol_Class'].values

# Turn data into tensors
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

# Make a copy to use later
X_full = X.clone()
y_full = y.clone()

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2,
                                                    random_state=2
)

nrows = X_train.size()[0]
ncols = X_train.size()[1]
X_tensor = X.clone().detach()
y_tensor = y.clone().detach().unsqueeze(1) # Shape: [1000, 1] for BCELoss
dataset = TensorDataset(X_tensor, y_tensor)
len(dataset)


C:\Users\MPennino\AppData\Local\Temp\ipykernel_11120\2301381008.py:32: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at D:\bld\libtorch_1784990676194\work\torch\csrc\utils\tensor_numpy.cpp:219.)
  y = torch.from_numpy(y).type(torch.float)


25460

In [5]:
batchsize = int(len(dataset) * 0.2)
batchsize

5092

In [6]:
# Load Test Dataset
import pickle
from torch.utils.data import DataLoader

with open(model_dir + testnames, "rb") as f:
    Test_datasets = pickle.load(f)

# Recreate your list of DataLoaders
recreated_loaders = [DataLoader(ds, batch_size=batchsize, shuffle=True) for ds in Test_datasets]

In [8]:
import torch
import torch.nn as nn

class ImprovedBinaryClassifier(nn.Module):
    def __init__(self, input_dim=ncols, hidden_dim=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.1),             # Prevents dead neurons
            nn.BatchNorm1d(hidden_dim),     # Stabilizes training
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.2),                # Prevents overfitting
            
            nn.Linear(hidden_dim, 1)        # Outputs raw logits (No Sigmoid here!)
        )
        
    def forward(self, x):
        return self.network(x)

# Load the PyTorch Model(s)

In [9]:
#modelpath = "/torch_models_5fold_future_NO3_sw.pth"

checkpoint = torch.load(model_dir + modelpath, weights_only=False)
len(checkpoint)


10

In [10]:
# 1. Recreate your blank list of models (must have the same architecture)
#loaded_models = [ImprovedBinaryClassifier(), ImprovedBinaryClassifier(), ImprovedBinaryClassifier(), ImprovedBinaryClassifier(), ImprovedBinaryClassifier()]

loaded_models = [ImprovedBinaryClassifier() for _ in range(len(checkpoint))]  # List comprehension for brevity

models = []
# 3. Restore the weights for each individual model
for i, model in enumerate(loaded_models):
    model.load_state_dict(checkpoint[f"model_{i}"])

    models.append(model)

    # 4. ALWAYS switch to evaluation mode if you are ready to test/predict!
    #model.eval()
    #print(i)
len(models), type(models[0])

(10, __main__.ImprovedBinaryClassifier)

# Model Evaluation Metrics

In [12]:
pcc_all = []
sensitivity_all = []
specificity_all = []

criterion = nn.BCEWithLogitsLoss()

with torch.no_grad():
    
    for i in range(len(models)):
        model = models[i]
        model.eval()  # Set the model to evaluation mode
        val_loader = recreated_loaders[i]
        val_loss = 0.0
        correct = 0
        total = 0
        # Convert the dataloader into a Python iterator
        data_iterator = iter(val_loader)

        # Grab the first mini-batch
        inputs, targets = next(data_iterator)

        #for inputs, targets in val_loader:  # Using the ith validation loader
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        val_loss += loss.item() * inputs.size(0)
        
        # Convert logits to binary predictions (0 or 1)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == targets).sum().item()
        total += targets.size(0)

        # Calculate True Positives, True Negatives, False Positives, False Negatives
        TP = torch.sum((preds == 1) & (targets == 1)).float()
        TN = torch.sum((preds == 0) & (targets == 0)).float()
        FP = torch.sum((preds == 1) & (targets == 0)).float()
        FN = torch.sum((preds == 0) & (targets == 1)).float()
        PCC = (TP + TN) / (TP + TN + FP + FN)
        sensitivity = TP / (TP + FN )
        specificity = TN / (TN + FP )
        #print(PCC, sensitivity, specificity)
        pcc_all.append(PCC)
        sensitivity_all.append(sensitivity)
        specificity_all.append(specificity)
        #print(len(preds), len(targets), len(inputs))

pcc_mean = np.mean(pcc_all)
sensitivity_mean = np.mean(sensitivity_all)
specificity_mean = np.mean(specificity_all)


#fold_acc = (sum(correct_all) / sum(total_all)) * 100
#fold_acc
#print(pcc_all)
print(f"PCC: {pcc_mean:.4f}")
print(f"Sensitivity (True Positives): {sensitivity_mean:.4f}")
print(f"Specificity (True Negatives): {specificity_mean:.4f}")

PCC: 0.8638
Sensitivity (True Positives): 0.8905
Specificity (True Negatives): 0.8371


In [13]:
# AUC Calculation
from sklearn.metrics import roc_auc_score
import numpy as np

auc_scores = []
with torch.no_grad():
    for i in range(len(models)):
        model = models[i]
        model.eval()  # Set the model to evaluation mode
        val_loader = recreated_loaders[i]
        val_loss = 0.0
        correct = 0
        total = 0
        # Convert the dataloader into a Python iterator
        data_iterator = iter(val_loader)

        # Grab the first mini-batch
        inputs, targets = next(data_iterator)
        
        with torch.inference_mode():
            preds = torch.round(torch.sigmoid(model(inputs))).squeeze()

            preds2 = preds.detach().cpu().tolist()
            target2 = targets.detach().cpu().tolist()


            # 2. Calculate the AUC Score
            auc_score = roc_auc_score(target2, preds2)
            auc_scores.append(auc_score)

auc_mean = np.mean(auc_scores)
print(f"Test AUC: {auc_mean:.4f}")

Test AUC: 0.8645


# Make Predictions

In [14]:
# Get list of predictors from training dataset
temp = input_data.drop(columns=['Viol_Class'])
names_list = temp.columns.tolist()
names_list

['PopDen2010Cat',
 'PctCrop2019Cat',
 'precip9120cat',
 'tmean9120cat',
 'HydrlCondCat',
 'RockNCat',
 'N_TW2012Cat',
 'N_Surp_kgsqkm_2017cat',
 'WtDepCat',
 'ElevCat',
 'Fe2O3Cat',
 'SandCat',
 'Hillslope_PctCat']

In [15]:
def generate_average_predictions(models, dataset, pred_data_):
    # Convert to Tensor
    PRED_DATA = dataset.values

    # Turn data into tensors
    PRED_DATA = torch.from_numpy(PRED_DATA).type(torch.float)

    all_predictions = []

    # Make Predictions on full HUC12 dataset, for prediction probabilities
    for model in models:
        model.eval()  # Set the model to evaluation mode
        with torch.inference_mode():
            y_probs = torch.sigmoid(model(PRED_DATA)).squeeze()
    
            # Convert from torch to pandas dataframe
            y_probs_df = pd.DataFrame(y_probs.cpu().numpy(), columns=['Pred_Viol_Prob'])

            all_predictions.append(y_probs_df)

    final_df = pd.concat(all_predictions, axis=1)
    # Calculate the average for each row
    
    preds_series = final_df.mean(axis=1)

    preds_df = preds_series.to_frame(name='Pred_Viol_Prob')

    # Merge with HUC12 
    final_preds = pd.concat([pred_data_['COMID'], preds_df], axis=1)

    return final_preds

# Get Confidence intervals on predictions

In [16]:
def generate_pred_CIs(models, dataset, pred_data_):
    # Convert to Tensor
    PRED_DATA = dataset.values

    # Turn data into tensors
    PRED_DATA = torch.from_numpy(PRED_DATA).type(torch.float)

    all_predictions = []

    # Make Predictions on full HUC12 dataset, for prediction probabilities
    for model in models:
        model.eval()  # Set the model to evaluation mode
        with torch.inference_mode():
            y_probs = torch.sigmoid(model(PRED_DATA)).squeeze()
    
            # Convert from torch to pandas dataframe
            y_probs_df = pd.DataFrame(y_probs.cpu().numpy(), columns=['Pred_Viol_Prob'])

            all_predictions.append(y_probs_df)

    final_df = pd.concat(all_predictions, axis=1)
    # Calculate the average for each row
    
    preds_series = final_df.mean(axis=1)
    
    preds_95ci_val_ = final_df.std(axis=1) *1.96 / np.sqrt(len(models))  # For 95% confidence interval
    preds_95ci_low_ = final_df.mean(axis=1) - final_df.std(axis=1) *1.96 / np.sqrt(len(models))  # For 95% confidence interval
    preds_95ci_high_ = final_df.mean(axis=1) + final_df.std(axis=1) *1.96 / np.sqrt(len(models))  # For 95% confidence interval

    preds_df = preds_series.to_frame(name='Pred_Viol_Prob')
    preds_95ci_val = preds_95ci_val_.to_frame(name='Pred_Viol_95ci_val')
    preds_95ci_low = preds_95ci_low_.to_frame(name='Pred_Viol_95ci_low')
    preds_95ci_high = preds_95ci_high_.to_frame(name='Pred_Viol_95ci_high')

    # Merge with HUC12 
    final_preds = pd.concat([pred_data_['COMID'], preds_df, preds_95ci_val, preds_95ci_low, preds_95ci_high], axis=1)

    return final_preds

# Get Standard Error on Predictions

In [17]:
def generate_pred_SE(models, dataset, pred_data_):
    # Convert to Tensor
    PRED_DATA = dataset.values

    # Turn data into tensors
    PRED_DATA = torch.from_numpy(PRED_DATA).type(torch.float)

    all_predictions = []

    # Make Predictions on full HUC12 dataset, for prediction probabilities
    for model in models:
        model.eval()  # Set the model to evaluation mode
        with torch.inference_mode():
            y_probs = torch.sigmoid(model(PRED_DATA)).squeeze()
    
            # Convert from torch to pandas dataframe
            y_probs_df = pd.DataFrame(y_probs.cpu().numpy(), columns=['Pred_Viol_Prob'])

            all_predictions.append(y_probs_df)

    final_df = pd.concat(all_predictions, axis=1)
    # Calculate the average for each row
    
    preds_mean_ = final_df.mean(axis=1)
    
    preds_se_ = final_df.std(axis=1) / np.sqrt(len(models))  # Standard error
 

    preds_df = preds_mean_.to_frame(name='Pred_Viol_Prob')
    preds_se = preds_se_.to_frame(name='Pred_Viol_SE')
    

    # Merge with HUC12 
    final_preds = pd.concat([pred_data_['COMID'], preds_df, preds_se], axis=1)

    return final_preds

# Scenario: Current Period (base year 2020)

In [27]:
# Load Prediction Dataset
dataset = "current_NO3_predictors_catchments.parquet"
pred_data_ = pd.read_parquet(future_dir+dataset) # Read a single Parquet file
#pred_data_ = pd.read_table(future_dir+dataset) # Read a single Parquet file

# Get list of predictors from training dataset
temp = input_data.drop(columns=['Viol_Class'])
names_list = temp.columns.tolist()
#print(names_list)

# Remove extra fields (model predictions won't work unless the prediction dataset has the same fields as the training dataset)
pred_data = pred_data_[names_list]
#pred_data.head(3)
#len(pred_data), pred_data['N_Surp_kgsqkm_2017ws'].mean() # for SW model
len(pred_data), pred_data['N_Surp_kgsqkm_2017cat'].mean() # for GW model

(2643994, np.float64(2495.0688943355635))

In [28]:
result2 = generate_average_predictions(models, pred_data, pred_data_)

print(f"Mean Prediction Probability: {result2['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result2['Pred_Viol_Prob'] > 0.5).sum() / len(result2):.2f}%")

Mean Prediction Probability: 0.2135, Violation Rate: 15.95%


In [30]:
# Save Dataset
#filename = 'torch_predictions_current_All_COMID_sw.parquet'
filename = 'torch_predictions_current_All_COMID_gw.parquet'

table = pa.Table.from_pandas(result2)
pq.write_table(table, future_dir + filename)

# Scenario: Current - 95% CI

In [25]:
result95 = generate_pred_CIs(models, pred_data, pred_data_)
result95.shape

(2643994, 5)

In [26]:
# Save Dataset
#filename = 'torch_predictions_current_95ci_COMID_sw.parquet'
filename = 'torch_predictions_current_95ci_COMID_gw.parquet'

table = pa.Table.from_pandas(result95)
pq.write_table(table, future_dir + filename)

# Scenario: Current Standard Error

In [23]:
resultSE = generate_pred_SE(models, pred_data, pred_data_)
resultSE.shape

(2643994, 3)

In [24]:
# Save Dataset
#filename = 'torch_predictions_current_SE_COMID_sw.parquet'
filename = 'torch_predictions_current_SE_COMID_gw.parquet'

table = pa.Table.from_pandas(resultSE)
pq.write_table(table, future_dir + filename)

# Scenario: 50% increase in N Surplus

In [32]:
pred_data_nsurp150 = pred_data.copy()
# For SW model
# pred_data_nsurp150['N_Surp_kgsqkm_2017ws'] = pred_data_nsurp150['N_Surp_kgsqkm_2017ws'] * 1.5
# pred_data_nsurp150['N_Surp_kgsqkm_2017ws'].mean(), pred_data['N_Surp_kgsqkm_2017ws'].mean()

# For GW model
pred_data_nsurp150['N_Surp_kgsqkm_2017cat'] = pred_data_nsurp150['N_Surp_kgsqkm_2017cat'] * 1.5
pred_data_nsurp150['N_Surp_kgsqkm_2017cat'].mean(), pred_data['N_Surp_kgsqkm_2017cat'].mean()

(np.float64(3742.6033415033453), np.float64(2495.0688943355635))

In [33]:

# # Convert to Tensor
# PRED_DATA = pred_data_nsurp.values

# # Turn data into tensors
# PRED_DATA = torch.from_numpy(PRED_DATA).type(torch.float)

# # Make Predictions on full HUC12 dataset, for prediction probabilities
# loaded_model.eval()
# with torch.inference_mode():
#     y_probs = torch.sigmoid(loaded_model(PRED_DATA)).squeeze()
    
# # Convert from torch to pandas dataframe
# y_probs_df = pd.DataFrame(y_probs.cpu().numpy(), columns=['Pred_Viol_Prob'])

# # Merge with HUC12 
# result2 = pd.concat([pred_data_['COMID'], y_probs_df], axis=1)
#result2['Pred_Viol_Prob'].mean(), 100 * (result2['Pred_Viol_Prob'] > 0.5).sum() / len(result2)
result2 = generate_average_predictions(models, pred_data_nsurp150, pred_data_)

print(f"Mean Prediction Probability: {result2['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result2['Pred_Viol_Prob'] > 0.5).sum() / len(result2):.2f}%")

Mean Prediction Probability: 0.2338, Violation Rate: 18.63%


In [34]:
# Save Dataset
#filename = 'torch_predictions_150perc_NSurp_COMID_sw.parquet'
filename = 'torch_predictions_150perc_NSurp_COMID_gw.parquet'

table = pa.Table.from_pandas(result2)
pq.write_table(table, future_dir + filename)

# Scenario: 50% Reduction in N Surplus

In [35]:
pred_data_nsurp50 = pred_data.copy()

# For SW model
# pred_data_nsurp50['N_Surp_kgsqkm_2017ws'] = pred_data_nsurp50['N_Surp_kgsqkm_2017ws'] * 0.5
# pred_data_nsurp50['N_Surp_kgsqkm_2017ws'].mean(), pred_data['N_Surp_kgsqkm_2017ws'].mean()

# For GW model
pred_data_nsurp50['N_Surp_kgsqkm_2017cat'] = pred_data_nsurp50['N_Surp_kgsqkm_2017cat'] * 0.5
pred_data_nsurp50['N_Surp_kgsqkm_2017cat'].mean(), pred_data['N_Surp_kgsqkm_2017cat'].mean()

(np.float64(1247.5344471677818), np.float64(2495.0688943355635))

In [36]:
# # Convert to Tensor
# PRED_DATA_SURP = pred_data_nsurp.values

# # Turn data into tensors
# PRED_DATA_SURP = torch.from_numpy(PRED_DATA_SURP).type(torch.float)

# # Make Predictions for prediction probabilities
# loaded_model.eval()
# with torch.inference_mode():
#     y_probs_surp = torch.sigmoid(loaded_model(PRED_DATA_SURP)).squeeze()

# # Convert to pd dataframe
# y_probs_surp_df = pd.DataFrame(y_probs_surp.cpu().numpy(), columns=['Pred_Viol_Prob'])

# # Merge with COMID 
# result_surp = pd.concat([pred_data_['COMID'], y_probs_surp_df], axis=1)
result_surp = generate_average_predictions(models, pred_data_nsurp50, pred_data_)

print(f"Mean Prediction Probability: {result_surp['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_surp['Pred_Viol_Prob'] > 0.5).sum() / len(result_surp):.2f}%")

Mean Prediction Probability: 0.1852, Violation Rate: 12.79%


In [37]:
# Save Results
#filename = 'torch_pred_scenario_50perc_NSurp_COMID_sw.parquet'
filename = 'torch_pred_scenario_50perc_NSurp_COMID_gw.parquet'

table = pa.Table.from_pandas(result_surp)
pq.write_table(table, future_dir + filename)

# Scenario: Complete removal of N Surplus


In [38]:
pred_data_nsurp = pred_data.copy()

# for SW model
# pred_data_nsurp['N_Surp_kgsqkm_2017ws'] = pred_data_nsurp['N_Surp_kgsqkm_2017ws'] * 0
# pred_data_nsurp['N_Surp_kgsqkm_2017ws'].mean(), pred_data['N_Surp_kgsqkm_2017ws'].mean()

# for GW model
pred_data_nsurp['N_Surp_kgsqkm_2017cat'] = pred_data_nsurp['N_Surp_kgsqkm_2017cat'] * 0
pred_data_nsurp['N_Surp_kgsqkm_2017cat'].mean(), pred_data['N_Surp_kgsqkm_2017cat'].mean()

(np.float64(0.0), np.float64(2495.0688943355635))

In [39]:
# # Convert to Tensor
# PRED_DATA_SURP = pred_data_nsurp.values

# # Turn data into tensors
# PRED_DATA_SURP = torch.from_numpy(PRED_DATA_SURP).type(torch.float)

# # Make Predictions for prediction probabilities
# loaded_model.eval()
# with torch.inference_mode():
#     y_probs_surp = torch.sigmoid(loaded_model(PRED_DATA_SURP)).squeeze()

# # Convert to pd dataframe
# y_probs_surp_df = pd.DataFrame(y_probs_surp.cpu().numpy(), columns=['Pred_Viol_Prob'])

# # Merge with COMID 
# result_surp = pd.concat([pred_data_['COMID'], y_probs_surp_df], axis=1)
result_surp = generate_average_predictions(models, pred_data_nsurp, pred_data_)

print(f"Mean Prediction Probability: {result_surp['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_surp['Pred_Viol_Prob'] > 0.5).sum() / len(result_surp):.2f}%")

Mean Prediction Probability: 0.1376, Violation Rate: 5.47%


In [40]:
# Save Results
#filename = 'torch_pred_scenario_0perc_NSurp_COMID_sw.parquet'
filename = 'torch_pred_scenario_0perc_NSurp_COMID_gw.parquet'

table = pa.Table.from_pandas(result_surp)
pq.write_table(table, future_dir + filename)

# Scenario: Increased NUE by 15%

In [42]:
filename = 'Scenario_NUE15_Dataset_Cat.parquet'
pred_data_nue_ = pq.read_table(future_dir + filename)
pred_data_nue1 = pred_data_nue_.to_pandas()
#pred_data_nue1.columns
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['COMID']) 
pred_data_nue = pred_data_nue1[names_list]
#pred_data_nue.shape, pred_data_nue['N_Surp_kgsqkm_2017ws'].mean()

In [52]:
# # Convert to Tensor
# PRED_DATA_NUE = pred_data_nue.values

# # Turn data into tensors
# PRED_DATA_NUE = torch.from_numpy(PRED_DATA_NUE).type(torch.float)

# # Make Predictions for prediction probabilities
# loaded_model.eval()
# with torch.inference_mode():
#     y_probs_nue = torch.sigmoid(loaded_model(PRED_DATA_NUE)).squeeze()

# # Convert to pd dataframe
# y_probs_nue_df = pd.DataFrame(y_probs_nue.cpu().numpy(), columns=['Pred_Viol_Prob'])

# # Merge with COMID 
# result_nue = pd.concat([pred_data_nue1['COMID'], y_probs_nue_df], axis=1)
#result_nue['Pred_Viol_Prob'].mean(), 100 * (result_nue['Pred_Viol_Prob'] > 0.5).sum() / len(result_nue)
result_nue15 = generate_average_predictions(models, pred_data_nue, pred_data_nue1)

print(f"Mean Prediction Probability: {result_nue15['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_nue15['Pred_Viol_Prob'] > 0.5).sum() / len(result_nue15):.2f}%")

Mean Prediction Probability: 0.1607, Violation Rate: 10.31%


In [ ]:
# Save Results
#filename = 'torch_pred_scenario_nue15_COMID_sw.parquet'
filename = 'torch_pred_scenario_nue15_COMID_gw.parquet'

table = pa.Table.from_pandas(result_nue15)
pq.write_table(table, future_dir + filename)

# Scenario: Increased NUE by 25%

In [47]:
filename = 'Scenario_NUE25_Dataset_Cat.parquet'
pred_data_nue_ = pq.read_table(future_dir + filename)
pred_data_nue1 = pred_data_nue_.to_pandas()
#pred_data_nue1.columns
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['COMID']) 
pred_data_nue = pred_data_nue1[names_list]
#pred_data_nue.shape, pred_data_nue['N_Surp_kgsqkm_2017ws'].mean()

In [48]:
# # Convert to Tensor
# PRED_DATA_NUE = pred_data_nue.values

# # Turn data into tensors
# PRED_DATA_NUE = torch.from_numpy(PRED_DATA_NUE).type(torch.float)

# # Make Predictions for prediction probabilities
# loaded_model.eval()
# with torch.inference_mode():
#     y_probs_nue = torch.sigmoid(loaded_model(PRED_DATA_NUE)).squeeze()

# # Convert to pd dataframe
# y_probs_nue_df = pd.DataFrame(y_probs_nue.cpu().numpy(), columns=['Pred_Viol_Prob'])

# # Merge with COMID 
# result_nue = pd.concat([pred_data_nue1['COMID'], y_probs_nue_df], axis=1)
#result_nue['Pred_Viol_Prob'].mean(), 100 * (result_nue['Pred_Viol_Prob'] > 0.5).sum() / len(result_nue)
result_nue = generate_average_predictions(models, pred_data_nue, pred_data_nue1)

print(f"Mean Prediction Probability: {result_nue['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_nue['Pred_Viol_Prob'] > 0.5).sum() / len(result_nue):.2f}%")

Mean Prediction Probability: 0.1988, Violation Rate: 14.79%


In [49]:
# Save Results
#filename = 'torch_pred_scenario_nue25_COMID_sw.parquet'
filename = 'torch_pred_scenario_nue25_COMID_gw.parquet'

table = pa.Table.from_pandas(result_nue)
pq.write_table(table, future_dir + filename)

# Scenario: Increased NUE by 50%

In [53]:
filename = 'Scenario_NUE50_Dataset_Cat.parquet'
pred_data_nue_ = pq.read_table(future_dir + filename)
pred_data_nue1 = pred_data_nue_.to_pandas()
#pred_data_nue1.columns
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['COMID']) 
pred_data_nue = pred_data_nue1[names_list]
#pred_data_nue.shape, pred_data_nue['N_Surp_kgsqkm_2017ws'].mean()

In [54]:
# # Convert to Tensor
# PRED_DATA_NUE = pred_data_nue.values

# # Turn data into tensors
# PRED_DATA_NUE = torch.from_numpy(PRED_DATA_NUE).type(torch.float)

# # Make Predictions for prediction probabilities
# loaded_model.eval()
# with torch.inference_mode():
#     y_probs_nue = torch.sigmoid(loaded_model(PRED_DATA_NUE)).squeeze()

# # Convert to pd dataframe
# y_probs_nue_df = pd.DataFrame(y_probs_nue.cpu().numpy(), columns=['Pred_Viol_Prob'])

# # Merge with COMID 
# result_nue = pd.concat([pred_data_nue1['COMID'], y_probs_nue_df], axis=1)
#result_nue['Pred_Viol_Prob'].mean(), 100 * (result_nue['Pred_Viol_Prob'] > 0.5).sum() / len(result_nue)
result_nue = generate_average_predictions(models, pred_data_nue, pred_data_nue1)
print(f"Mean Prediction Probability: {result_nue['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_nue['Pred_Viol_Prob'] > 0.5).sum() / len(result_nue):.2f}%")

Mean Prediction Probability: 0.1607, Violation Rate: 10.31%


In [55]:
# Save Results
#filename = 'torch_pred_scenario_nue50_COMID_sw.parquet'
filename = 'torch_pred_scenario_nue50_COMID_gw.parquet'

table = pa.Table.from_pandas(result_nue)
pq.write_table(table, future_dir + filename)

# Scenario: Increased Productivity (55% increase, constant NUE)

In [60]:
#pred_data_prod = pred_data.copy()
#pred_data_prod['n_surplus_kgsqkm'] = pred_data_prod['n_surplus_kgsqkm'] * 2
#pred_data_prod['n_surplus_kgsqkm'].mean(), pred_data['n_surplus_kgsqkm'].mean()

filename = 'Scenario_Prod55_NUE0_Dataset_Cat.parquet'
pred_data_prod_ = pq.read_table(future_dir + filename)
pred_data_prod1 = pred_data_prod_.to_pandas()
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['HUC12']) 
pred_data_prod = pred_data_prod1[names_list]
pred_data_prod.shape

(2644024, 13)

In [61]:
# # Convert to Tensor
# PRED_DATA_PROD = pred_data_prod.values

# # Turn data into tensors
# PRED_DATA_PROD = torch.from_numpy(PRED_DATA_PROD).type(torch.float)

# # Make Predictions for prediction probabilities
# loaded_model.eval()
# with torch.inference_mode():
#     y_probs_prod = torch.sigmoid(loaded_model(PRED_DATA_PROD)).squeeze()

# # Convert to pd dataframe
# y_probs_prod_df = pd.DataFrame(y_probs_prod.cpu().numpy(), columns=['Pred_Viol_Prob'])

# # Merge with HCOMID
# result_prod = pd.concat([pred_data_prod1['COMID'], y_probs_prod_df], axis=1)

result_prod = generate_average_predictions(models, pred_data_prod, pred_data_prod1)

print(f"Mean Prediction Probability: {result_prod['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_prod['Pred_Viol_Prob'] > 0.5).sum() / len(result_prod):.2f}%")

Mean Prediction Probability: 0.2172, Violation Rate: 16.47%


In [62]:
# Save Results
#filename = 'torch_pred_scenario_prod55_nue0_COMID_sw.parquet'
filename = 'torch_pred_scenario_prod55_nue0_COMID_gw.parquet'

table = pa.Table.from_pandas(result_prod)
pq.write_table(table, future_dir + filename)

# Scenario: Increased Productivity & NUE (55% increase, 15 Increase NUE)
* With Decreased Inputs

In [64]:
filename = 'Scenario_Prod55_NUE15_Dataset_Cat.parquet'
pred_data_prod_ = pq.read_table(future_dir + filename)
pred_data_prodnue1 = pred_data_prod_.to_pandas()
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['HUC12']) 
pred_data_prodnue = pred_data_prodnue1[names_list]
pred_data_prodnue.shape

(2644024, 13)

In [65]:
# print means of all variables
for col in pred_data_prodnue.columns:
    print(f"Mean of {col}: {pred_data_prodnue[col].mean()}")

Mean of PopDen2010Cat: 36.6628915885058
Mean of PctCrop2019Cat: 13.710048543689322
Mean of precip9120cat: 914.0705914530853
Mean of tmean9120cat: 11.784710677773267
Mean of HydrlCondCat: 22.789452658878844
Mean of RockNCat: 188.20016148269505
Mean of N_TW2012Cat: 6.959639596629984
Mean of N_Surp_kgsqkm_2017cat: 1999.267250773162
Mean of WtDepCat: 143.4242017378287
Mean of ElevCat: 669.3709864114269
Mean of Fe2O3Cat: 6.735817336811901
Mean of SandCat: 32.21361576976921
Mean of Hillslope_PctCat: 10.266098450349705


In [66]:
# # Convert to Tensor
# PRED_DATA_PRODNUE = pred_data_prodnue.values

# # Turn data into tensors
# PRED_DATA_PRODNUE = torch.from_numpy(PRED_DATA_PRODNUE).type(torch.float)

# # Make Predictions for prediction probabilities
# loaded_model.eval()
# with torch.inference_mode():
#     y_probs_prodnue = torch.sigmoid(loaded_model(PRED_DATA_PRODNUE)).squeeze()

# # Convert to pd dataframe
# y_probs_prodnue_df = pd.DataFrame(y_probs_prodnue.cpu().numpy(), columns=['Pred_Viol_Prob'])

# # Merge with COMID
# result_prodnue = pd.concat([pred_data_prodnue1['COMID'], y_probs_prodnue_df], axis=1)

result_prodnue = generate_average_predictions(models, pred_data_prodnue, pred_data_prodnue1)
print(f"Mean Prediction Probability: {result_prodnue['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_prodnue['Pred_Viol_Prob'] > 0.5).sum() / len(result_prodnue):.2f}%")

Mean Prediction Probability: 0.2086, Violation Rate: 15.56%


In [67]:
# Save Results
#filename = 'torch_pred_scenario_prod55_nue15_COMID_sw.parquet'
filename = 'torch_pred_scenario_prod55_nue15_COMID_gw.parquet'

table = pa.Table.from_pandas(result_prodnue)
pq.write_table(table, future_dir + filename)

# Scenario: Increased Productivity & NUE (55% increase, 15 Increase NUE)
* With Constant Inputs

In [69]:
filename = 'Scenario_Prod55_NUE15_constInput_Dataset_Cat.parquet'
pred_data_prod_ = pq.read_table(future_dir + filename)
pred_data_prodnue1 = pred_data_prod_.to_pandas()
pred_data_prodnue2 = pred_data_prodnue1[names_list]

result_prodnue = generate_average_predictions(models, pred_data_prodnue2, pred_data_prodnue1)
print(f"Mean Prediction Probability: {result_prodnue['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_prodnue['Pred_Viol_Prob'] > 0.5).sum() / len(result_prodnue):.2f}%")

Mean Prediction Probability: 0.2019, Violation Rate: 14.99%


In [70]:
# Save Results
#filename = 'torch_pred_scenario_prod55_nue15_constInput_COMID_sw.parquet'
filename = 'torch_pred_scenario_prod55_nue15_constInput_COMID_gw.parquet'

table = pa.Table.from_pandas(result_prodnue)
pq.write_table(table, future_dir + filename)

# Scenario: Future projection

In [574]:
input_data.columns

Index(['PopDen2010Ws', 'PctForest2019Ws', 'PctCrop2019Ws', 'precip9120ws',
       'tmean9120ws', 'BFIWs', 'permws', 'RockNWs', 'N_TW2012Ws',
       'N_Surp_kgsqkm_2017ws', 'ElevWs', 'Fe2O3Ws', 'NHDslope_Pct_Ws',
       'Viol_Class'],
      dtype='str')

In [71]:
# Load Dataset
filename = 'Dataset_Future_Catchment_4.5GISS.parquet'
#filename = 'Dataset_Future_Catchment_8.5GISS.parquet'
#filename = 'Dataset_Future_Catchment_4.5Hadgem.parquet'
#filename = 'Dataset_Future_Catchment_8.5Hadgem.parquet'

# Get list of predictors from training dataset
temp = input_data.drop(columns=['Viol_Class'])
names_list = temp.columns.tolist()

pred_data_fut_ = pq.read_table(future_dir + filename)
pred_data_fut1 = pred_data_fut_.to_pandas()
pred_data_fut1.shape
pred_data_fut1.columns
pred_data_fut = pred_data_fut1[names_list]
pred_data_fut.shape

(2644024, 13)

In [72]:
# print means of all variables
for col in pred_data_fut.columns:
    print(f"Mean of {col}: {pred_data_fut[col].mean()}")

Mean of PopDen2010Cat: 44.88014516865042
Mean of PctCrop2019Cat: 16.287791410647298
Mean of precip9120cat: 995.5507542118638
Mean of tmean9120cat: 12.79468021442703
Mean of HydrlCondCat: 22.789452658878844
Mean of RockNCat: 188.20016148269505
Mean of N_TW2012Cat: 5.4935571606308296
Mean of N_Surp_kgsqkm_2017cat: 2062.5402728137283
Mean of WtDepCat: 143.4242017378287
Mean of ElevCat: 669.3709864114269
Mean of Fe2O3Cat: 6.735817336811901
Mean of SandCat: 32.21361576976921
Mean of Hillslope_PctCat: 10.266098450349705


In [73]:
# # Convert to Tensor
# PRED_DATA_FUT = pred_data_fut.values

# # Turn data into tensors
# PRED_DATA_FUT = torch.from_numpy(PRED_DATA_FUT).type(torch.float)

# # Make Predictions for prediction probabilities
# loaded_model.eval()
# with torch.inference_mode():
#     y_probs_fut = torch.sigmoid(loaded_model(PRED_DATA_FUT)).squeeze()

# # Convert to pd dataframe
# y_probs_fut_df = pd.DataFrame(y_probs_fut.cpu().numpy(), columns=['Pred_Viol_Prob'])

# # Merge with Catchment 
# result_fut = pd.concat([pred_data_fut1['COMID'], y_probs_fut_df], axis=1)
# #result_fut.shape, result_fut['Pred_Viol_Prob'].mean(), 100 * (result_fut['Pred_Viol_Prob'] > 0.5).sum() / len(result_fut)

result_fut = generate_average_predictions(models, pred_data_fut, pred_data_fut1)

print(f"Mean Prediction Probability: {result_fut['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_fut['Pred_Viol_Prob'] > 0.5).sum() / len(result_fut):.2f}%")

Mean Prediction Probability: 0.1684, Violation Rate: 11.18%


In [74]:
# Save Results
#filename = 'torch_pred_scenario_fut_RCP4.5G_COMID_sw.parquet'
#filename = 'torch_pred_scenario_fut_RCP8.5G_COMID_sw.parquet'
#filename = 'torch_pred_scenario_fut_RCP4.5H_COMID_sw.parquet'
#filename = 'torch_pred_scenario_fut_RCP8.5H_COMID_sw.parquet'

filename = 'torch_pred_scenario_fut_RCP4.5G_COMID_gw.parquet'


table = pa.Table.from_pandas(result_fut)
pq.write_table(table, future_dir + filename)

# Scenario: Future Projection & Increased Prod & Constant NUE

In [75]:
filename = 'Scenario_RCP45G_Prod55_NUE0_Dataset_Cat.parquet'
pred_data_fut_ = pq.read_table(future_dir + filename)
pred_data_fut1_ = pred_data_fut_.to_pandas()
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['HUC12']) 
pred_data_futprod = pred_data_fut1_[names_list]
pred_data_futprod.shape
# print means of all variables
for col in pred_data_futprod.columns:
    print(f"Mean of {col}: {pred_data_futprod[col].mean()}")

Mean of PopDen2010Cat: 44.88014516865042
Mean of PctCrop2019Cat: 16.287791410647298
Mean of precip9120cat: 995.5507542118638
Mean of tmean9120cat: 12.79468021442703
Mean of HydrlCondCat: 22.789452658878844
Mean of RockNCat: 188.20016148269505
Mean of N_TW2012Cat: 5.4935571606308296
Mean of N_Surp_kgsqkm_2017cat: 2807.8365317449316
Mean of WtDepCat: 143.4242017378287
Mean of ElevCat: 669.3709864114269
Mean of Fe2O3Cat: 6.735817336811901
Mean of SandCat: 32.21361576976921
Mean of Hillslope_PctCat: 10.266098450349705


In [76]:
# Generate predictions
result_fut = generate_average_predictions(models, pred_data_futprod, pred_data_fut1_)

print(f"Mean Prediction Probability: {result_fut['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_fut['Pred_Viol_Prob'] > 0.5).sum() / len(result_fut):.2f}%")

Mean Prediction Probability: 0.1755, Violation Rate: 11.75%


In [77]:
# Save Results
#filename = 'torch_pred_scenario_fut_RCP45G_prod55_nue0_COMID_sw.parquet'
filename = 'torch_pred_scenario_fut_RCP45G_prod55_nue0_COMID_gw.parquet'

table = pa.Table.from_pandas(result_fut)
pq.write_table(table, future_dir + filename)

# Scenario: Future Projection & Increased Prod55 & NUE15 (Decreased Inputs)

In [78]:
filename = 'Scenario_RCP45G_Prod55_NUE15_Dataset_Cat.parquet'
pred_data_fut_ = pq.read_table(future_dir + filename)
pred_data_fut1_ = pred_data_fut_.to_pandas()
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['HUC12']) 
pred_data_fut2 = pred_data_fut1_[names_list]
pred_data_fut2.shape

# print means of all variables
for col in pred_data_fut2.columns:
    print(f"Mean of {col}: {pred_data_fut2[col].mean()}")

Mean of PopDen2010Cat: 44.88014516865042
Mean of PctCrop2019Cat: 16.287791410647298
Mean of precip9120cat: 995.5507542118638
Mean of tmean9120cat: 12.79468021442703
Mean of HydrlCondCat: 22.789452658878844
Mean of RockNCat: 188.20016148269505
Mean of N_TW2012Cat: 5.4935571606308296
Mean of N_Surp_kgsqkm_2017cat: 1709.1899087936158
Mean of WtDepCat: 143.4242017378287
Mean of ElevCat: 669.3709864114269
Mean of Fe2O3Cat: 6.735817336811901
Mean of SandCat: 32.21361576976921
Mean of Hillslope_PctCat: 10.266098450349705


In [79]:
# generate predictions

result_fut = generate_average_predictions(models, pred_data_fut2, pred_data_fut1_)

print(f"Mean Prediction Probability: {result_fut['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_fut['Pred_Viol_Prob'] > 0.5).sum() / len(result_fut):.2f}%")

Mean Prediction Probability: 0.1669, Violation Rate: 11.21%


In [80]:
# Save Results
#filename = 'torch_pred_scenario_fut_RCP45G_prod55_nue15_COMID_sw.parquet'
filename = 'torch_pred_scenario_fut_RCP45G_prod55_nue15_COMID_gw.parquet'

table = pa.Table.from_pandas(result_fut)
pq.write_table(table, future_dir + filename)

# Scenario: Future Projection & Increased Prod55 & NUE15 (Constant Inputs)

In [81]:
filename = 'Scenario_RCP45G_Prod55_NUE15_constInput_Dataset_Cat.parquet'
pred_data_fut_ = pq.read_table(future_dir + filename)
pred_data_fut1_ = pred_data_fut_.to_pandas()
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['HUC12']) 
pred_data_fut2b = pred_data_fut1_[names_list]

result_fut = generate_average_predictions(models, pred_data_fut2b, pred_data_fut1_)
print(f"Mean Prediction Probability: {result_fut['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_fut['Pred_Viol_Prob'] > 0.5).sum() / len(result_fut):.2f}%")

Mean Prediction Probability: 0.1711, Violation Rate: 11.46%


In [82]:
# Save Results
#filename = 'torch_pred_scenario_fut_RCP45G_prod55_nue15_constInput_COMID_sw.parquet'
filename = 'torch_pred_scenario_fut_RCP45G_prod55_nue15_constInput_COMID_gw.parquet'

table = pa.Table.from_pandas(result_fut)
pq.write_table(table, future_dir + filename)

# Scenario: Future Projection & Increased Prod55 & NUE25 (Decreased Inputs)

In [83]:
filename = 'Scenario_RCP45G_Prod55_NUE25_Dataset_Cat.parquet'
pred_data_fut_ = pq.read_table(future_dir + filename)
pred_data_fut1_ = pred_data_fut_.to_pandas()
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['HUC12']) 
pred_data_fut3 = pred_data_fut1_[names_list]
pred_data_fut3.shape

# print means of all variables
for col in pred_data_fut3.columns:
    print(f"Mean of {col}: {pred_data_fut3[col].mean()}")

Mean of PopDen2010Cat: 44.88014516865042
Mean of PctCrop2019Cat: 16.287791410647298
Mean of precip9120cat: 995.5507542118638
Mean of tmean9120cat: 12.79468021442703
Mean of HydrlCondCat: 22.789452658878844
Mean of RockNCat: 188.20016148269505
Mean of N_TW2012Cat: 5.4935571606308296
Mean of N_Surp_kgsqkm_2017cat: 1303.8333734736648
Mean of WtDepCat: 143.4242017378287
Mean of ElevCat: 669.3709864114269
Mean of Fe2O3Cat: 6.735817336811901
Mean of SandCat: 32.21361576976921
Mean of Hillslope_PctCat: 10.266098450349705


In [84]:
result_fut = generate_average_predictions(models, pred_data_fut3, pred_data_fut1_)

print(f"Mean Prediction Probability: {result_fut['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_fut['Pred_Viol_Prob'] > 0.5).sum() / len(result_fut):.2f}%")

Mean Prediction Probability: 0.1574, Violation Rate: 10.01%


In [85]:
# Save Results
#filename = 'torch_pred_scenario_fut_RCP45G_prod55_nue25_COMID_sw.parquet'
filename = 'torch_pred_scenario_fut_RCP45G_prod55_nue25_COMID_gw.parquet'

table = pa.Table.from_pandas(result_fut)
pq.write_table(table, future_dir + filename)

# Scenario: Future Projection & Increased Prod55 & NUE50 (Decreasd Inputs)

In [86]:
filename = 'Scenario_RCP45G_Prod55_NUE50_Dataset_Cat.parquet'
pred_data_fut_ = pq.read_table(future_dir + filename)
pred_data_fut1_ = pred_data_fut_.to_pandas()
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['HUC12']) 
pred_data_fut3 = pred_data_fut1_[names_list]
pred_data_fut3.shape

# print means of all variables
for col in pred_data_fut3.columns:
    print(f"Mean of {col}: {pred_data_fut3[col].mean()}")

Mean of PopDen2010Cat: 44.88014516865042
Mean of PctCrop2019Cat: 16.287791410647298
Mean of precip9120cat: 995.5507542118638
Mean of tmean9120cat: 12.79468021442703
Mean of HydrlCondCat: 22.789452658878844
Mean of RockNCat: 188.20016148269505
Mean of N_TW2012Cat: 5.4935571606308296
Mean of N_Surp_kgsqkm_2017cat: 665.4719151888581
Mean of WtDepCat: 143.4242017378287
Mean of ElevCat: 669.3709864114269
Mean of Fe2O3Cat: 6.735817336811901
Mean of SandCat: 32.21361576976921
Mean of Hillslope_PctCat: 10.266098450349705


In [87]:
result_fut = generate_average_predictions(models, pred_data_fut3, pred_data_fut1_)

print(f"Mean Prediction Probability: {result_fut['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_fut['Pred_Viol_Prob'] > 0.5).sum() / len(result_fut):.2f}%")

Mean Prediction Probability: 0.1288, Violation Rate: 6.42%


In [88]:
# Save Results
#filename = 'torch_pred_scenario_fut_RCP45G_prod55_nue50_COMID_sw.parquet'
filename = 'torch_pred_scenario_fut_RCP45G_prod55_nue50_COMID_gw.parquet'

table = pa.Table.from_pandas(result_fut)
pq.write_table(table, future_dir + filename)

# Comparing N Surplus Averages (SW)

In [90]:
# value0 = pred_data['N_Surp_kgsqkm_2017ws'].mean()
# value1 = pred_data_fut2['N_Surp_kgsqkm_2017ws'].mean()
# value2 = pred_data_futprod['N_Surp_kgsqkm_2017ws'].mean()
# value3 = pred_data_fut['N_Surp_kgsqkm_2017ws'].mean()
# value4 = pred_data_prodnue['N_Surp_kgsqkm_2017ws'].mean()
# value5 = pred_data_prod['N_Surp_kgsqkm_2017ws'].mean()

# value6 = pred_data_prodnue2['N_Surp_kgsqkm_2017ws'].mean()
# value7 = pred_data_fut2b['N_Surp_kgsqkm_2017ws'].mean()

# value9 = pred_data_nsurp50['N_Surp_kgsqkm_2017ws'].mean()
# value10 = pred_data_nsurp150['N_Surp_kgsqkm_2017ws'].mean()


# print(f"{value0:.2f} = Average N Surplus for Base Scenario")
# print(f"{value5:.2f} = Average N Surplus for Prod55_NUE0 Scenario")
# print(f"{value4:.2f} = Average N Surplus for Prod55_NUE15 Scenario")
# print(f"{value6:.2f} = Average N Surplus for Prod55_NUE15_ConstInput Scenario")
# print(f"{value3:.2f} = Average N Surplus for RCP45G_Prod0_NUE0 Scenario")
# print(f"{value2:.2f} = Average N Surplus for RCP45G_Prod55_NUE0 Scenario")
# print(f"{value1:.2f} = Average N Surplus for RCP45G_Prod55_NUE15 Scenario")
# print(f"{value7:.2f} = Average N Surplus for RCP45G_Prod55_NUE15_ConstInput Scenario")

# print(f"{value9:.2f} = Average N Surplus for 50% Scenario")
# print(f"{value10:.2f} = Average N Surplus for 150% Scenario")



# Comparing N Surplus Averages (GW)

In [92]:
value0 = pred_data['N_Surp_kgsqkm_2017cat'].mean()
value1 = pred_data_fut2['N_Surp_kgsqkm_2017cat'].mean()
value2 = pred_data_futprod['N_Surp_kgsqkm_2017cat'].mean()
value3 = pred_data_fut['N_Surp_kgsqkm_2017cat'].mean()
value4 = pred_data_prodnue['N_Surp_kgsqkm_2017cat'].mean()
value5 = pred_data_prod['N_Surp_kgsqkm_2017cat'].mean()

value6 = pred_data_prodnue2['N_Surp_kgsqkm_2017cat'].mean()
value7 = pred_data_fut2b['N_Surp_kgsqkm_2017cat'].mean()

value9 = pred_data_nsurp50['N_Surp_kgsqkm_2017cat'].mean()
value10 = pred_data_nsurp150['N_Surp_kgsqkm_2017cat'].mean()


print(f"{value0:.2f} = Average N Surplus for Base Scenario")
print(f"{value9:.2f} = Average N Surplus for 50% Scenario")
print(f"{value10:.2f} = Average N Surplus for 150% Scenario")

print(f"{value5:.2f} = Average N Surplus for Prod55_NUE0 Scenario")
print(f"{value4:.2f} = Average N Surplus for Prod55_NUE15 Scenario")
print(f"{value6:.2f} = Average N Surplus for Prod55_NUE15_ConstInput Scenario")
print(f"{value3:.2f} = Average N Surplus for RCP45G_Prod0_NUE0 Scenario")
print(f"{value2:.2f} = Average N Surplus for RCP45G_Prod55_NUE0 Scenario")
print(f"{value1:.2f} = Average N Surplus for RCP45G_Prod55_NUE15 Scenario")
print(f"{value7:.2f} = Average N Surplus for RCP45G_Prod55_NUE15_ConstInput Scenario")



2495.07 = Average N Surplus for Base Scenario
1247.53 = Average N Surplus for 50% Scenario
3742.60 = Average N Surplus for 150% Scenario
3321.87 = Average N Surplus for Prod55_NUE0 Scenario
1999.27 = Average N Surplus for Prod55_NUE15 Scenario
1469.16 = Average N Surplus for Prod55_NUE15_ConstInput Scenario
2062.54 = Average N Surplus for RCP45G_Prod0_NUE0 Scenario
2807.84 = Average N Surplus for RCP45G_Prod55_NUE0 Scenario
1709.19 = Average N Surplus for RCP45G_Prod55_NUE15 Scenario
2196.23 = Average N Surplus for RCP45G_Prod55_NUE15_ConstInput Scenario


In [93]:
# Print current time at the end
from datetime import datetime
print("Finished at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Finished at: 2026-09-15 14:40:38
